# PyHealth ReXKG Pipeline

This notebook shows a PyHealth-native ReXKG workflow similar to other examples:

1. Load dataset
2. Set tasks
3. Build sample datasets



## 1) Configure Path and Install needed libraries and import RexKG pyhealth implementation

In [ ]:
# Make sure to install the needed libraries used for the rexkg PyHealth files. 
# Kernal for this conda virtual environment is running 3.13.13

# !pip uninstall -y tokenizers transformers
# !pip install -U pip setuptools wheel

# !pip install -q \
#   neraug==0.1.1 \
#   pandas numpy tqdm scikit-learn scipy statsmodels seqeval \
#   sentencepiece safetensors tensorboardX
#!python -m pip install neraug
#!python -m pip install torch
#!python -m pip uninstall -y openai
#!python -m pip install openai==0.28

#TODO: create the folder structure for pipeline

from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "PyHealth").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root. Expected a parent directory containing 'PyHealth/'."
    )

# Set `PROJECT_ROOT` to your repo root if auto-detection does not match your environment.
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if not (PROJECT_ROOT / "PyHealth").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

rexkg_dir = PROJECT_ROOT / "PyHealth" / "examples" / "rexkg"
folders_to_create = [
    rexkg_dir / "data",
    rexkg_dir / "data" / "data_split",
    rexkg_dir / "result" / "local_run" / "run_entity",
    rexkg_dir / "result" / "local_run" / "run_relation",
    rexkg_dir / "result" / "local_run" / "entities",
    rexkg_dir / "result" / "local_run" / "relation",
    rexkg_dir / "result" / "local_run" / "kg",
]



for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)


print("ReXKG folder structure is ready at:", rexkg_dir)

In [ ]:
# import importlib.util
from pyhealth.datasets import RexKGDataset, RexKGCheXpertDataset 
# from pyhealth.models import RexKG
from pyhealth.tasks import (
    RexKGEntityExtractionRadiology,
    RexKGRelationExtractionRadiology,
    RexKGReverseStructureRadiology,
    RexKGGetEntitiesRadiology,
    RexKGGPT4EntityExtractionRadiology,
    RexKGGPT4RelationExtractionRadiology,
    RexKGStructureData,
    RexKGUMLS
)

PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth


## 2) Declare the dataset CheXpert Dataset
The CheXpert dataset is used in this full pipe line example. 

Filename, `df_chexpert_plus_240401.csv` should be located in ./data/ directory of this python notebook and can be downloaded at:
https://stanfordaimi.azurewebsites.net/datasets/5158c524-d3ab-4e02-96e9-6ee9efc110a1

The MIMC CXR 2.0.0 dataset can also be used and downloaded at:
https://physionet.org/content/mimic-cxr/2.0.0/

You may have to create and account and accept Terms of Agreement to access. 

In [7]:
chexpert_csv = PROJECT_ROOT / "examples" / "rexkg" / "data" / "df_chexpert_plus_240401.csv"
if not chexpert_csv.exists():
    raise FileNotFoundError(f"Missing CheXpert CSV: {chexpert_csv}")

cheXpert = RexKGCheXpertDataset(
    root=str(chexpert_csv),
    table=[
        "path_to_image",
        "path_to_dcm",
        "frontal_lateral",
        "ap_pa",
        "deid_patient_id",
        "patient_report_date_order",
        "report",
        "section_narrative",
        "section_clinical_history",
        "section_history",
        "section_comparison",
        "section_technique",
        "section_procedure_comments",
        "section_findings",
        "section_impression",
        "section_end_of_impression",
        "section_summary",
        "section_accession_number",
        "age",
        "sex",
        "race",
        "ethnicity",
        "interpreter_needed",
        "insurance_type",
        "recent_bmi",
        "deceased",
        "split",
    ],
    dev=False,
 )

json_gpt4_entity_save_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "gpt4_entities_chexpert_plus.json"



No config path provided, using default RexKG CheXpert config
Initializing rexkg_chexpert dataset from /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/data (dev mode: False)
No cache_dir provided. Using default cache dir: /home/strawhat/.cache/pyhealth/eed028e6-5e09-5bae-bb10-e5ceb87da255


# 3) Use Chat GPT 4 to label the extracted Entities and Relations 

An **Entity** in the schema are categorized into six types as listed.
1. Anatomy: anatomical structures within the body.
2. Disorder: any abnormal findings or diseases identified within radiology reports.
3. Concept: descriptors used to modify other entities, for example, ”acute”, ”severe”, and ”increasing”.
4. Device: any instrument or apparatus used for medical purposes, for example, “tube”, “clip”, “wire”.
5. Procedure: medical procedures used to diagnose, measure, monitor, or treat conditions, such as “sternotomy”.
6. Size: measurements of disorders or anatomical structures, for example, “3-mm”.

A **Relation** is defined as a directed edge between two entities. Following the previous work (Jain et al. 2021a), our schema uses three relations as listed.
1. Suggestive of: source entity (e.g., findings) may suggest the presence of the target entity (e.g., a disease).
2. Located at: source entity is located at the target entity.
3. Modify: source entity modifies or provides additional information about the target entity.

In [ ]:

json_gpt4_entity_save_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "gpt4_entities_chexpert_plus.json"

# You may have to create an OpenAI instance on Azure and Deploy a Model in Microsoft Foundary to fill these fields properly
api_key=""
api_base="https://azureopenai-instance-uiuc.cognitiveservices.azure.com/"
api_type = "azure"
api_version = "2024-12-01-preview"
model = "gpt-4o"


res = RexKGGPT4EntityExtractionRadiology.set_task(
    dataset=cheXpert,  # RexKGCheXpertDataset
    save_json_file=str(json_gpt4_entity_save_path),
    start_idx=0,
    end_idx=1000, # set extraction to 1000 records so building the Entity & Relation models an achievable size on local machine
    api_key=api_key,
    api_base=api_base,
    api_type = api_type,
    api_version = api_version,
    model=model,
)

Extracting patient entity  0  out of  1000  records.
Extracting patient entity  100  out of  1000  records.
Extracting patient entity  200  out of  1000  records.
Extracting patient entity  300  out of  1000  records.
hello??
input json: [{'role': 'system', 'content': "You are a radiologist performing clinical term extraction from the FINDINGS and IMPRESSION sections in the radiology report.                     Here a clinical term can be in ['anatomy','disorder_present','disorder_notpresent','procedures','devices','concept', 'devices_present','devices_notpresent','size'].                     'anatomy' refers to the anatomical body;                    'disorder_present' refers to findings or diseases are present according to the sentence;                     'disorder_notpresent' refers to findings or diseases are not present according to the sentence;                     'procedures' refers to procedures are used to diagnose, measure, monitor or treat problems;                     'de

In [9]:
json_gpt4_entity_relation_save_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "gpt4_entities_relations_chexpert_plus.json"
post_proccess_json_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "gpt4_entities_relations_chexpert_plus_post.json"


res = RexKGGPT4RelationExtractionRadiology.set_task(
    input_json_file=str(json_gpt4_entity_save_path),
    save_json_file=str(json_gpt4_entity_relation_save_path),
    post_proccess_json=str(post_proccess_json_path),
    api_key=api_key,
    api_base=api_base,
    api_type = api_type,
    api_version = api_version,
    model=model,
)

# Have to skip some records due to the filter policy OpenAI and Azure impose. See link below for more details.
# https://learn.microsoft.com/en-us/azure/foundry-classic/foundry-models/concepts/content-filter?tabs=user-prompt

Extracting patient relation  0  out of  1000
Extracting patient relation  100  out of  1000
Extracting patient relation  200  out of  1000
Extracting patient relation  300  out of  1000
SKIP filtered records (does not adhere to OpenAI's Safety Standards)  train/patient10907/study2/view2_lateral.jpg
Extracting patient relation  400  out of  1000
using input record  500  out of 1000
input_json:  {'PA and lateral chest show a right apical pneumothorax has filled in with fluid since the prior study.': {'PA': 'procedures', 'lateral': 'concept', 'chest': 'anatomy', 'right': 'concept', 'apical': 'anatomy', 'pneumothorax': 'disorder_notpresent', 'fluid': 'disorder_present', 'prior': 'concept', 'study': 'procedures'}, 'There is linear atelectasis at the left lung base.': {'linear': 'concept', 'atelectasis': 'disorder_present', 'left': 'concept', 'lung': 'anatomy', 'base': 'anatomy'}, 'There is elevation of the right hemidiaphragm, unchanged with blunting of the lateral and posterior right costo


## 4) PURE Format conversion
structure_data.py

In [10]:
save_train_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "data_split" / "train.json"
save_test_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "data_split" / "test.json"

structure_result = RexKGStructureData.set_task(
    post_proccess_json=str(post_proccess_json_path),
    save_train_path=str(save_train_path),
    save_test_path=str(save_test_path),
)

100%|██████████| 892/892 [00:00<00:00, 5102.79it/s]


No pneumothorax.
No acute osseous abnormalities.
1001 sentences done
No evidence of pneumothorax.
There is no visible pneumothorax.
However, cannot exclude the possibility of infection.
2001 sentences done
There has been cholecystectomy.
No pneumothorax is seen.
No pneumothorax.
No pleural effusion.
No pneumothorax.
No pneumothorax.
No osseous abnormalities.
There are no effusions.
No visible pneumothorax.
There is minimal blunting of the left costophrenic angle on the AP film, but no corresponding pleural effusion is seen on the lateral view.
3001 sentences done
No definite pneumothorax.
4001 sentences done
There is no pneumothorax.
No pneumothorax.
5001 sentences done


## 5) Load RexKGDataset - Data Preperation

In [11]:
# dataset = RexKGDataset(root=str(expected))
train_dataset = RexKGDataset(root=str(save_train_path))
test_dataset = RexKGDataset(root=str(save_test_path))
dev_dataset = RexKGDataset(root=str(save_test_path))

## 6) Train the NER/Entity Natural Language Processing (NLP) model 

Use BERT based DL model

In [12]:
entity_task = RexKGEntityExtractionRadiology()


# Force output under this notebook folder.
entity_output_dir = PROJECT_ROOT / "examples" / "rexkg" / "result" / "local_run"  / "run_entity"
model = RexKGEntityExtractionRadiology.set_task(
    train_data=train_dataset,
    dev_data=dev_dataset,
    test_data=test_dataset,
    model="bert-base-uncased",
    output_dir=str(entity_output_dir),
    do_train=True,
    do_eval=True,
    eval_test=True,
    learning_rate=1e-5,
    task_learning_rate=5e-4,
    train_batch_size=8,
    eval_batch_size=64,
    num_epoch=1,
    context_window=5,
)
print(model)

pred_file = entity_output_dir / "ent_pred_mimic_headct.json"
print("Expected prediction file:", pred_file)
print("Prediction file exists:", pred_file.exists())


Some weights of BertForEntity were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['ner_classifier.0.network.0.bias', 'ner_classifier.0.network.0.weight', 'ner_classifier.0.network.3.bias', 'ner_classifier.0.network.3.weight', 'ner_classifier.1.bias', 'ner_classifier.1.weight', 'width_embedding.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyError: 'device_present'

## 7) Train the NER/Relation NLP model

Use BERT based DL model

All input/output paths below stay inside `PyHealth/examples/rexkg/`.

In [ ]:
# relations model directory
relation_output_dir = PROJECT_ROOT / "examples" / "rexkg" / "result" / "local_run" / "run_relation"

# Auto-detection walks up from relation_output_dir; set explicitly if it fails.
# NER_SRC_DIR = PROJECT_ROOT / "src" / "ner"

relation_metrics = RexKGRelationExtractionRadiology.set_task(
    train_file=str(save_train_path),
    entity_output_dir=str(entity_output_dir),
    entity_predictions_dev="ent_pred_mimic_headct.json",
    entity_predictions_test="ent_pred_mimic_headct.json",
    model="bert-base-uncased",
    output_dir=str(relation_output_dir),
    do_train=True,
    do_eval=True,
    eval_with_gold=True,
    do_lower_case=True,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    context_window=20,
    max_seq_length=256,
    ner_src_dir=str(PROJECT_ROOT),
)
print(relation_metrics)

relation_pred_file = relation_output_dir / "predictions.json"
print("Expected relation prediction file:", relation_pred_file)
print("Relation prediction file exists:", relation_pred_file.exists())

# 8) Construct Reverse Knowledge Graph
Build entity/relation tables from reversed JSON

converts the relation‑extraction outputs back into the structured, report‑level format that the KG construction pipeline expects -i.e., it reverses the preprocessing that turned raw reports into PURE training/test examples so the predicted entities/relations can be used for node and edge construction. 

In [ ]:
predictions_path = PROJECT_ROOT / "examples" / "rexkg" /"result"/ "local_run" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")


structured_output_path = PROJECT_ROOT / "examples" / "rexkg" / "data" / "your_test_file.json"
processed_docs = RexKGReverseStructureRadiology.set_task(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))

print("\ 10 lines of structured output JSON:")
with structured_output_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if 2231 <= i <= 2282:
            print(f"{i:04d}: {line.rstrip()}")
        if i > 2282:
            break

## 9) Get Entities 
Generate count based CSV artifacts by aggregating Entity frequencies by normalized text and type. Then normalize Entity typing rules, and force measurement-like tokens into size and dropping noisy digit-containing non-size entities.


In [ ]:
# Build entity and relation summary CSV files from Step 5 output JSON.
entity_csv_output_dir = PROJECT_ROOT / "examples" / "rexkg" / "result" / "local_run" / "entities"
relation_csv_output_dir = PROJECT_ROOT / "examples" / "rexkg" / "result" / "local_run" / "relation"

get_entities_result = RexKGGetEntitiesRadiology.set_task(
    ent_pred_mimic_headct=str(structured_output_path),
    ent_real_pred_mimic_headct=str(structured_output_path),
    save_entity_dir=str(entity_csv_output_dir),
    save_real_dir=str(relation_csv_output_dir),
)

print(get_entities_result)
print("Entity CSV folder:", entity_csv_output_dir)
print("Relation CSV folder:", relation_csv_output_dir)
print("all_entities.csv exists:", (entity_csv_output_dir / "all_entities.csv").exists())
print("all_relations.csv exists:", (relation_csv_output_dir / "all_relations.csv").exists())

## 10) Get UMLS 

In [ ]:

                                # output             input
# !python get_umls_entities.py --save_entity_dir ../result/local_run/entities
save_umls_dir = PROJECT_ROOT / "examples" / "rexkg" / "result" / "local_run" / "entities"

umls = RexKGUMLS.set_task(
        input_dir=entity_csv_output_dir,
        out_dir=save_umls_dir
    )

## 11) Filter CUI

In [ ]:
# !python filter_cui.py --save_entity_dir ../result/local_run/entities

## 12) Structure Entities

In [ ]:
# !python structure_entities.py --save_entity_dir ../result/local_run/entities --ignore_count 1



## 13) get kg nodes

In [ ]:
# !python get_kg_nodes.py \
#   --save_entity_dir ../result/local_run/entities \
#   --save_real_dir ../result/local_run/relation \
#   --save_kg_dir ../result/local_run/kg



## 14) get size of relations

In [ ]:
# !python get_size_relations.py \
#   --entity_dir ../result/local_run/entities \
#   --real_dir ../result/local_run/relation


## 15) get inference

this should be in metrics

In [ ]:

# python get_inference_data.py

# !ls -lh /content/drive/MyDrive/cs598_project/src/kg_construct/result/local_run/kg

# Extra Credit
16) Generate the ReXKG Scores and show a part of the knowledge graph